# Cross-Industry Accelerator — Create Data Agents
### Auto-Create Fabric IQ QA and Operations Agents

This notebook creates **two agents** backed by the ontology and data sources:

| Agent | Purpose | Power |
|---|---|---|
| **QA Agent** | Answer ad-hoc data questions in natural language | Queries Lakehouse + Eventhouse via ontology |
| **Ops Agent** | Monitor operations and surface alerts/anomalies | Real-time event analysis via ontology |

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. Resolves the ontology item ID from the workspace
3. Creates the QA Agent item linked to the ontology
4. Creates the Ops Agent item linked to the ontology
5. Configures each agent with industry-appropriate instructions

> **Prerequisites:**
> 1. Run `05_Create_Ontology.ipynb` to create the ontology first
> 2. Run `04_Create_Semantic_Model.ipynb` to create the semantic model
> 3. Data must be loaded (Lakehouse + Eventhouse) for agents to query
> 4. Each industry must have a `{Industry}_Agent_Instructions.ipynb` notebook that defines `QA_AGENT_INSTRUCTIONS` and `OPS_AGENT_INSTRUCTIONS`. If the notebook is missing, generic instructions are used.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD INDUSTRY-SPECIFIC AGENT INSTRUCTIONS
# ════════════════════════════════════════════════════════════════════════
# Calls {INDUSTRY_LABEL}_Agent_Instructions via notebookutils.notebook.run().
# The child notebook returns instructions as JSON via notebook.exit().
# Falls back to generic instructions if the notebook is missing.

import json

_instructions_nb = f"{INDUSTRY_LABEL}_Agent_Instructions"
print(f"Loading agent instructions from: {_instructions_nb}")

try:
    _result = notebookutils.notebook.run(_instructions_nb, 600)
    _parsed = json.loads(_result)
    QA_AGENT_INSTRUCTIONS = _parsed["qa"]
    OPS_AGENT_INSTRUCTIONS = _parsed["ops"]
    QA_FEWSHOTS = _parsed.get("qa_fewshots", {})
    OPS_FEWSHOTS = _parsed.get("ops_fewshots", {})
    print(f"  QA instructions:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
    print(f"  Ops instructions: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
    print(f"  QA fewshots:  {sum(len(v) for v in QA_FEWSHOTS.values())} queries across {len(QA_FEWSHOTS)} datasources")
    print(f"  Ops fewshots: {sum(len(v) for v in OPS_FEWSHOTS.values())} queries across {len(OPS_FEWSHOTS)} datasources")
except Exception as e:
    print(f"  Could not load {_instructions_nb}: {e}")
    print(f"  Will use generic instructions.")
    QA_AGENT_INSTRUCTIONS = None
    OPS_AGENT_INSTRUCTIONS = None
    QA_FEWSHOTS = {}
    OPS_FEWSHOTS = {}

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESOLVE WORKSPACE ITEMS
# ════════════════════════════════════════════════════════════════════════

import sempy.fabric as fabric
import json, requests

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
items_df = fabric.list_items()

# Resolve Ontology item ID
ont_matches = items_df[(items_df["Type"] == "Ontology") & (items_df["Display Name"] == ONTOLOGY_NAME)]
if ont_matches.empty:
    print(f"⚠ Ontology '{ONTOLOGY_NAME}' not found. Run 05_Create_Ontology first.")
    print(f"Available ontologies:")
    print(items_df[items_df["Type"] == "Ontology"][["Display Name", "Id"]].to_string(index=False))
    ontology_item_id = None
else:
    ontology_item_id = str(ont_matches.iloc[0].Id)
    print(f"✓ Ontology: {ONTOLOGY_NAME} → {ontology_item_id}")

# Resolve Lakehouse item ID
lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
lakehouse_item_id = str(lh_matches.iloc[0].Id) if not lh_matches.empty else None
print(f"✓ Lakehouse: {LAKEHOUSE_NAME} → {lakehouse_item_id}")

# Resolve Eventhouse item ID
eventhouse_item_id = None
if EVENTHOUSE_NAME:
    eh_matches = items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == EVENTHOUSE_NAME)]
    if not eh_matches.empty:
        eventhouse_item_id = str(eh_matches.iloc[0].Id)
        print(f"✓ Eventhouse: {EVENTHOUSE_NAME} → {eventhouse_item_id}")

print(f"\n  QA Agent name:  {DATA_AGENT_NAME}")
print(f"  Ops Agent name: {OPS_AGENT_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INSTALL FABRIC DATA AGENT SDK (requires kernel restart after install)
# ════════════════════════════════════════════════════════════════════════
# Run this cell ONCE, then restart the kernel, then skip this cell
# and continue from the next cell.

import subprocess, sys

# Force upgrade typing_extensions to system path
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--upgrade", "--force-reinstall", "--target",
    "/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages",
    "typing_extensions>=4.12", "-q"
])

# Install the SDK
subprocess.check_call([sys.executable, "-m", "pip", "install", "fabric-data-agent-sdk", "-q"])

print("Installed. RESTART THE KERNEL now, then run the next cell.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# IMPORT SDK (run after kernel restart)
# ════════════════════════════════════════════════════════════════════════

from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
    delete_data_agent,
)

print("fabric-data-agent-sdk imported successfully.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE QA AGENT + ADD DATA SOURCES (via fabric-data-agent-sdk)
# ════════════════════════════════════════════════════════════════════════

import time, requests as req_lib
import sempy.fabric as fabric

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
api_headers = {"Authorization": f"Bearer {access_token}"}
BASE = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# Clean up any existing agents with the same names
items_df = fabric.list_items()
for agent_name in [DATA_AGENT_NAME, OPS_AGENT_NAME]:
    deleted = False
    # Try SDK delete first (cleanest approach)
    try:
        delete_data_agent(agent_name)
        print(f"  Deleted existing agent '{agent_name}' via SDK.")
        deleted = True
        time.sleep(5)
    except Exception:
        pass
    # Fallback: delete via REST API if found in items list
    if not deleted:
        matches = items_df[items_df["Display Name"] == agent_name]
        if not matches.empty:
            for _, row in matches.iterrows():
                item_id = str(row["Id"])
                dr = req_lib.delete(f"{BASE}/items/{item_id}", headers=api_headers, timeout=30)
                print(f"  Deleted '{agent_name}' via REST: HTTP {dr.status_code}")
                time.sleep(5)
                deleted = True
    if not deleted:
        print(f"  '{agent_name}' — no existing agent found (clean slate).")

time.sleep(3)

# Create QA Agent
print(f"\nCreating QA Agent: {DATA_AGENT_NAME}...")
qa_agent = None
try:
    qa_agent = create_data_agent(DATA_AGENT_NAME)
    print(f"  Created: {DATA_AGENT_NAME}")
except Exception as e:
    print(f"  Create: {e}")
    if "already in use" in str(e).lower():
        try:
            qa_agent = FabricDataAgentManagement(DATA_AGENT_NAME)
            print(f"  Connected to existing.")
        except Exception as e2:
            print(f"  Connect: {e2}")

if qa_agent:
    # Use industry-specific instructions if provided by the agent instructions notebook,
    # otherwise fall back to a generic prompt.
    if 'QA_AGENT_INSTRUCTIONS' in dir() and QA_AGENT_INSTRUCTIONS:
        qa_instructions = QA_AGENT_INSTRUCTIONS
    else:
        qa_instructions = (
            f"You are a data analyst agent for the {INDUSTRY} industry. "
            f"Answer questions about {INDUSTRY} data using the connected data sources. "
            f"Query dimension tables (dim_*) for reference data and fact tables (fact_*) for metrics. "
            f"Use the Warehouse for SQL queries, the KQL Database for real-time event data, "
            f"and the Semantic Model for pre-built measures. "
            f"You are scoped to {INDUSTRY} data only."
        )
    qa_agent.update_configuration(instructions=qa_instructions)
    print(f"  Instructions set ({len(qa_instructions)} chars).")

    for ds_name, ds_type in [
        (WAREHOUSE_NAME, "warehouse"),
        (SEMANTIC_MODEL_NAME, "semanticmodel"),
        (LAKEHOUSE_NAME, "lakehouse"),
    ]:
        print(f"  Adding {ds_type}: {ds_name}...")
        try:
            qa_agent.add_datasource(ds_name, type=ds_type)
            print(f"    Added.")
        except Exception as e:
            print(f"    {e}")

    if EVENTHOUSE_DATABASE:
        print(f"  Adding kqldatabase: {EVENTHOUSE_DATABASE}...")
        try:
            qa_agent.add_datasource(EVENTHOUSE_DATABASE, type="kqldatabase")
            print(f"    Added.")
        except Exception as e:
            print(f"    {e}")

    # Upload per-datasource example queries (fewshots)
    if 'QA_FEWSHOTS' in dir() and QA_FEWSHOTS:
        for ds_name, fewshots in QA_FEWSHOTS.items():
            if fewshots:
                print(f"  Uploading {len(fewshots)} example queries for {ds_name}...")
                try:
                    qa_agent.upload_fewshots(ds_name, fewshots)
                    print(f"    Uploaded.")
                except Exception as e:
                    print(f"    Fewshots: {e}")
    else:
        print(f"  No per-datasource example queries (QA_FEWSHOTS not set).")

        pass

    print(f"  Publishing...")    except Exception:

    try:        print(f"  Sources: {qa_agent.get_datasources()}")

        qa_agent.publish()        print(f"  Config: {qa_agent.get_configuration()}")

        print(f"  Published: {DATA_AGENT_NAME}")    try:

    except Exception as e:
        print(f"  {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE OPS AGENT + ADD DATA SOURCES (via fabric-data-agent-sdk)
# ════════════════════════════════════════════════════════════════════════

print(f"\nCreating Ops Agent: {OPS_AGENT_NAME}...")
try:
    ops_agent = create_data_agent(OPS_AGENT_NAME)
    print(f"  Created: {OPS_AGENT_NAME}")
except Exception as e:
    if "already" in str(e).lower():
        print(f"  Agent already exists. Connecting to existing...")
        ops_agent = FabricDataAgentManagement(OPS_AGENT_NAME)
    else:
        print(f"  Error: {e}")
        ops_agent = None

if ops_agent:
    # Use industry-specific instructions if provided by the agent instructions notebook,
    # otherwise fall back to a generic prompt.
    if 'OPS_AGENT_INSTRUCTIONS' in dir() and OPS_AGENT_INSTRUCTIONS:
        ops_instructions = OPS_AGENT_INSTRUCTIONS
    else:
        ops_instructions = (
            f"You are an operations monitoring agent for the {INDUSTRY} industry. "
            f"Analyze event streams and fact tables to detect anomalies and surface alerts. "
            f"Focus on KPIs, SLA compliance, quality metrics, and risk indicators. "
            f"Use the Warehouse for historical trends and the KQL Database for real-time events. "
            f"When reporting issues, provide severity and recommended actions. "
            f"You are scoped to {INDUSTRY} operational data only."
        )
    ops_agent.update_configuration(instructions=ops_instructions)
    print(f"  Instructions set ({len(ops_instructions)} chars).")

    # Add Warehouse as data source
    print(f"  Adding Warehouse: {WAREHOUSE_NAME}...")
    try:
        ops_agent.add_datasource(WAREHOUSE_NAME, type="warehouse")
        print(f"  Warehouse added.")
    except Exception as e:
        print(f"  Warehouse: {e}")

    # Add Eventhouse/KQL Database as data source
    if EVENTHOUSE_DATABASE:
        print(f"  Adding KQL Database: {EVENTHOUSE_DATABASE}...")
        try:
            ops_agent.add_datasource(EVENTHOUSE_DATABASE, type="kqldatabase")
            print(f"  KQL Database added.")
        except Exception as e:
            print(f"  KQL DB: {e}")

    # Upload per-datasource example queries (fewshots)
    if 'OPS_FEWSHOTS' in dir() and OPS_FEWSHOTS:
        for ds_name, fewshots in OPS_FEWSHOTS.items():
            if fewshots:
                print(f"  Uploading {len(fewshots)} example queries for {ds_name}...")
                try:
                    ops_agent.upload_fewshots(ds_name, fewshots)

                    print(f"    Uploaded.")        print(f"  Publish: {e}")

                except Exception as e:    except Exception as e:

                    print(f"    Fewshots: {e}")        print(f"  Ops Agent published: {OPS_AGENT_NAME}")

    else:        ops_agent.publish()

        print(f"  No per-datasource example queries (OPS_FEWSHOTS not set).")    try:

    print(f"  Publishing...")
    # Publish the agent

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# AGENT CREATION SUMMARY
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print(f"DATA AGENT SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  Ontology:       {ONTOLOGY_NAME}")
print(f"  Lakehouse:      {LAKEHOUSE_NAME}")
print(f"  Eventhouse:     {EVENTHOUSE_NAME or 'N/A'}")
print(f"")
print(f"  QA Agent:       {DATA_AGENT_NAME}")
print(f"    → Answers ad-hoc data questions about {INDUSTRY} data")
print(f"    → Queries: dimensions, facts, aggregates")
print(f"")
print(f"  Ops Agent:      {OPS_AGENT_NAME}")
print(f"    → Monitors real-time events and operational metrics")
print(f"    → Surfaces: anomalies, SLA breaches, quality alerts")
print(f"")
print(f"✅ Agent creation complete.")
print(f"   Next: Run 07_Create_Dashboards.ipynb to create real-time and Power BI dashboards.")